![corrective-RAG flow](./corrective-rag.png)

**References**
- [Corrective RAG Paper (USTC, UCLA, Google DeepMind)](https://arxiv.org/pdf/2401.15884)
- [Tavily Official Docs - LangChain Integration](https://docs.tavily.com/documentation/integrations/langchain)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embedding_function = OpenAIEmbeddings(model="text-embedding-3-large")

vector_store = Chroma(
    embedding_function=embedding_function,
    collection_name="income_tax_collection",
    persist_directory="./income_tax_collection",
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph


# context가 vector store에서 가져온 결과 or tavily websearch의 결과이기 때문에 List[Document] -> list 로 변경
class AgentState(TypedDict):
    query: str
    # context: List[Document]
    context: list
    answer: str


graph_builder = StateGraph(AgentState)

In [ ]:
def retrieve(state: AgentState):
    query = state["query"]
    docs = retriever.invoke(query)
    return {"context": docs}

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

In [ ]:
from langsmith import Client
from langchain_core.output_parsers import StrOutputParser

client = Client()
generate_prompt = client.pull_prompt(
    "rlm/rag-prompt", dangerously_pull_public_prompt=True
)


def generate(state: AgentState):
    context = state["context"]
    query = state["query"]
    rag_chain = generate_prompt | llm | StrOutputParser()
    response = rag_chain.invoke({"question": query, "context": context})
    return {"answer": response}

In [ ]:
from typing import Literal

doc_relevance_prompt = client.pull_prompt(
    "langchain-ai/rag-document-relevance", dangerously_pull_public_prompt=True
)


def check_doc_relevance(state: AgentState) -> Literal["relevant", "irrelevant"]:
    query = state["query"]
    context = state["context"]
    print(f"context == {context}")
    doc_relevance_chain = doc_relevance_prompt | llm
    response = doc_relevance_chain.invoke({"question": query, "documents": context})
    print(f"doc relevance response == {response}")

    if response["Score"] == 1:
        return "relevant"
    return "irrelevant"

In [ ]:
from langchain_core.prompts import PromptTemplate

# web search를 하기 좋은 질문으로 바꾸는 prompt
rewrite_prompt = PromptTemplate.from_template("""
사용자의 질문을 보고, 웹검색에 용이하게 사용자의 질문을 변경해주세요.
질문: {query}
""")


# rewrite node
def rewrite(state: AgentState):
    query = state["query"]
    rewrite_chain = rewrite_prompt | llm | StrOutputParser()

    response = rewrite_chain.invoke({"query": query})
    print(f"rewrite question == {response}")
    return {"query": response}

- https://docs.tavily.com/documentation/integrations/langchain

In [ ]:
# %pip install -U langchain-tavily

In [ ]:
from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(
    max_results=3,
    topic="general",
    # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
)


def web_search(state: AgentState):
    query = state["query"]
    results = tavily_search_tool.invoke(query)
    print(f"web_search result == {results}")
    return {"context": results}

In [ ]:
# add nodes

graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("generate", generate)
graph_builder.add_node("rewrite", rewrite)
graph_builder.add_node("web_search", web_search)

In [ ]:
from langgraph.graph import START, END

graph_builder.add_edge(START, "retrieve")
graph_builder.add_conditional_edges(
    "retrieve", check_doc_relevance, {"relevant": "generate", "irrelevant": "rewrite"}
)
graph_builder.add_edge("rewrite", "web_search")
graph_builder.add_edge("web_search", "generate")
graph_builder.add_edge("generate", END)

In [ ]:
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# test
initial_state = {"query": "연봉이 5천만원인 거주자의 소득세는 얼마인가요?"}
graph.invoke(initial_state)

In [ ]:
# test with irrelevant question
initial_state = {"query": "여의도 맛집은?"}
graph.invoke(initial_state)

# Corrective RAG vs Self-RAG

The key difference lies in **how each approach handles retrieval failure**.

**Self-RAG** focuses on **self-reflection after generation**:
- If retrieved documents are irrelevant → stops (END)
- After generating an answer → checks for **hallucination** and **helpfulness**
- If hallucinated → regenerates / If unhelpful → rewrites query and retries retrieval

**Corrective RAG** focuses on **correcting the retrieval itself**:
- If retrieved documents are irrelevant → **rewrites the query** for web search → **searches the web (Tavily)** → generates answer
- Does not give up on retrieval failure — instead, it falls back to external knowledge via web search

In short, Self-RAG asks *"Is my answer good?"* while Corrective RAG asks *"Can I find better sources?"*